<a href="https://colab.research.google.com/github/martatolos/eae-dsaa/blob/main/mlops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLOps: Full Lifecycle with MLflow

> **Goal:** Experience the complete MLOps lifecycle — data extraction, preprocessing, model training with experiment tracking, model registry, inference, drift detection, and REST serving — using a real Spotify tracks dataset and MLflow.

> **Requirements:** All dependencies are installed in the first cell. No external accounts needed.

> **Important:** Run cells top-to-bottom in a single Colab session. Variables set in earlier sections are used by later ones.


## Section 0: Environment Setup

### 0.1 Install Dependencies

Run this cell first. It installs all packages needed for this lab.

In [ ]:
%pip install mlflow==3.12.0 polars evidently flask pyngrok --quiet

In [ ]:
import os
import subprocess
import threading
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import polars as pl
import requests
from evidently import Report
from evidently.presets import DataDriftPreset
from flask import Flask, jsonify
from flask import request as flask_request
from IPython.display import HTML, display
from mlflow import MlflowClient
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

### 0.2 Clone Repo and Set Working Directory

We clone the repository so we have access to the bundled Spotify dataset in `data/`.

In [ ]:
!git clone https://github.com/martatolos/eae-dsaa.git /content/eae-dsaa --quiet

BASE_DIR = Path("/content/eae-dsaa")
DATA_DIR = BASE_DIR / "data"
os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Dataset exists: {(DATA_DIR / 'spotify_tracks.csv').exists()}")

### 0.3 Start MLflow Server

MLflow needs a **SQLite backend** (not just a file store) to support the Model Registry feature we use in Section 5. We start it as a background process and poll until it responds.

> **Why SQLite?** MLflow's Model Registry (versioning, aliases) requires a database. SQLite is built into Python — no extra installation needed.

In [ ]:
mlflow_process = subprocess.Popen(
    [
        "mlflow",
        "server",
        "--backend-store-uri",
        "sqlite:///mlflow.db",
        "--default-artifact-root",
        str(BASE_DIR / "mlruns"),
        "--host",
        "0.0.0.0",
        "--port",
        "5000",
        "--allowed-hosts",
        "*",
        "--cors-allowed-origins",
        "*",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until server responds (MLflow root / returns 200 when ready)
for i in range(20):
    try:
        r = requests.get("http://localhost:5000/", timeout=2)
        if r.status_code == 200:
            print(f"MLflow server ready after {(i + 1) * 2}s")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("MLflow server did not start in time — check the process")

### 0.4 Get MLflow UI URL

Colab's built-in port proxy gives you a public URL for any local port — no account or token needed. The cell below generates a URL for the MLflow server we started in 0.3.

If you're running this notebook locally (not in Colab), it falls back to `http://localhost:5000`.


In [ ]:
try:
    from google.colab.output import eval_js

    MLFLOW_PUBLIC_URL = eval_js("google.colab.kernel.proxyPort(5000)")
except ImportError:
    MLFLOW_PUBLIC_URL = "http://localhost:5000"

print(f"MLflow UI: {MLFLOW_PUBLIC_URL}")
print("Open this URL in a new tab — you should see an empty MLflow Experiments page.")

### 0.5 Configure MLflow Tracking URI

In [ ]:
mlflow.set_tracking_uri(MLFLOW_PUBLIC_URL)
# Verify: list experiments (returns empty list on a fresh server)
client = mlflow.MlflowClient()
experiments = client.search_experiments()
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiments: {len(experiments)} (expected 0 on a fresh server)")

## Section 1: Data Extraction

### Why Data Extraction is a Distinct Step

In production ML pipelines, data extraction is always separated from preprocessing:
- **Auditability**: you can always reprocess from the raw source
- **Reproducibility**: the raw file is immutable; every transform is tracked
- **Data lake pattern**: raw data lands in one place, processed data in another

Here we load from our bundled CSV (committed to the repo), mimicking a read from a data lake.

**Polars `scan_csv` vs `read_csv`:** `scan_csv()` returns a `LazyFrame` — data is NOT loaded into memory yet. Operations you chain on it are recorded as a query plan. Only `.collect()` executes the plan. For our 5K-row dataset the difference is small, but the pattern scales well.

In [ ]:
# scan_csv = lazy (returns LazyFrame, no data loaded yet)
raw_lf = pl.scan_csv(DATA_DIR / "spotify_tracks.csv")

# collect() executes the query plan and materialises the DataFrame
raw_df = raw_lf.collect()

print(f"Shape: {raw_df.shape}")
print("\nSchema:")
print(raw_df.schema)
print("\nFirst 5 rows:")
raw_df.head()

In [ ]:
# Save raw data (simulating writing to a data lake landing zone)
raw_df.write_csv(DATA_DIR / "raw_data.csv")
print(f"Saved raw_data.csv ({raw_df.shape[0]} rows)")

## Section 1.5: Exploratory Data Analysis

### Why EDA Before Preprocessing

Before writing a single transform, we need to understand:
- Is `popularity` (our target) normally distributed or skewed?
- Which features correlate with popularity?
- Are features on very different scales? (Hint: yes — `loudness` is in dB, `duration_ms` is in milliseconds)
- How balanced is the genre distribution? (Dataset has 6 genres, balanced at ~900 tracks each)

These observations directly inform our preprocessing choices.

In [ ]:
# Work in pandas for visualization convenience
eda_df = raw_df.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Target distribution
axes[0].hist(eda_df["popularity"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Popularity Distribution")
axes[0].set_xlabel("Popularity (0-100)")
axes[0].set_ylabel("Track Count")

# Genre distribution
genre_counts = eda_df["genre"].value_counts()
axes[1].barh(genre_counts.index, genre_counts.values, color="steelblue")
axes[1].set_title("Tracks per Genre")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with popularity (bar chart)
numeric_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "explicit",
    "year",
]

corr = eda_df[numeric_cols + ["popularity"]].corr()["popularity"].drop("popularity").sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["salmon" if v < 0 else "steelblue" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.set_title("Feature Correlation with Popularity")
ax.set_xlabel("Pearson Correlation")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

top_positive = corr.tail(2).index.tolist()
print(f"Top positive correlations: {top_positive}")

In [ ]:
# Scatter plots: top 2 correlated features vs popularity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, feat in enumerate(top_positive):
    axes[i].scatter(eda_df[feat], eda_df["popularity"], alpha=0.2, s=5, color="steelblue")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("popularity")
    axes[i].set_title(f"{feat} vs popularity")
plt.tight_layout()
plt.show()

In [ ]:
# Feature scale comparison — this is why we normalize
print("Feature ranges (note the very different scales):")
print(eda_df[["loudness", "duration_ms", "tempo", "danceability"]].describe().round(2))
print()
print(eda_df.describe().round(2))

## Section 2: Preprocessing

### Preprocessing Pipeline

Preprocessing transforms raw data into model-ready features. The same transformations **must** be applied consistently at both training and inference time — inconsistency here is one of the most common bugs in production ML.

We use Polars' lazy API to chain all transforms, then `.collect()` once to execute.

**One-hot vs ordinal encoding for genre:** Genre is a **nominal** categorical variable — there is no inherent order between pop, rock, and hip-hop. We use one-hot encoding (one binary column per genre, drop-first) rather than ordinal encoding (which would imply a false numerical ordering). Tree-based models handle many binary features well.

In [ ]:
# Reload raw data as a LazyFrame
raw_lf = pl.scan_csv(DATA_DIR / "raw_data.csv")

# Step 1: Encode binary column
# Step 2: Normalise loudness and duration_ms to [0, 1]
# Compute min/max on the full dataset (in production, fit on train set only)
raw_collected = raw_lf.collect()
LOUDNESS_MIN = float(raw_collected["loudness"].min())
LOUDNESS_MAX = float(raw_collected["loudness"].max())
DURATION_MIN = float(raw_collected["duration_ms"].min())
DURATION_MAX = float(raw_collected["duration_ms"].max())

preprocessed_lf = raw_lf.with_columns(
    [
        pl.col("explicit").cast(pl.Int8),
        ((pl.col("loudness") - LOUDNESS_MIN) / (LOUDNESS_MAX - LOUDNESS_MIN)).alias("loudness"),
        ((pl.col("duration_ms") - DURATION_MIN) / (DURATION_MAX - DURATION_MIN)).alias("duration_ms"),
    ]
)

print("Normalisation params saved for inference:")
print(f"  loudness : [{LOUDNESS_MIN:.2f}, {LOUDNESS_MAX:.2f}]")
print(f"  duration_ms: [{DURATION_MIN:.0f}, {DURATION_MAX:.0f}]")

In [ ]:
# Step 3: One-hot encode genre (via pandas get_dummies, then back to Polars)
preprocessed_df = preprocessed_lf.collect().to_pandas()
genre_dummies = pd.get_dummies(preprocessed_df["genre"], prefix="genre", drop_first=True, dtype=int)
GENRE_COLUMNS = list(genre_dummies.columns)

preprocessed_df = pd.concat([preprocessed_df.drop(columns=["genre"]), genre_dummies], axis=1)

# Convert back to Polars
preprocessed_df_pl = pl.from_pandas(preprocessed_df)

print(f"Genre columns created ({len(GENRE_COLUMNS)}): {GENRE_COLUMNS}")
print(f"\nProcessed shape: {preprocessed_df_pl.shape}")

In [ ]:
# Save processed data
preprocessed_df_pl.write_csv(DATA_DIR / "processed_data.csv")
print(f"Saved processed_data.csv {preprocessed_df_pl.shape}")

preprocessed_df_pl.head(3)

**Preprocessing artifacts stored as notebook variables for use in later sections:**
- `LOUDNESS_MIN`, `LOUDNESS_MAX` — for normalising new loudness values at inference time
- `DURATION_MIN`, `DURATION_MAX` — for normalising new duration values at inference time
- `GENRE_COLUMNS` — the ordered list of one-hot column names (must match exactly at inference)
- `preprocessed_df` — the pandas DataFrame used as Evidently reference in Section 6

## Section 3: Data Preparation

### Config-Driven Feature Selection

In production, feature lists and split parameters live in config files — not hardcoded in training scripts. This means swapping features is a config change, not a code change.

> **Note:** `GENRE_COLUMNS` was set in Section 2. This section must run in the same kernel session as Section 2.

In [ ]:
# CONFIG uses GENRE_COLUMNS from Section 2 (must run after Section 2)
CONFIG = {
    "features": [
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo",
        "duration_ms",
        "explicit",
        "year",
        *GENRE_COLUMNS,  # one-hot genre columns from Section 2
    ],
    "target": "popularity",
    "test_size": 0.2,
    "random_state": 42,
}

print(f"Features ({len(CONFIG['features'])}):")
for f in CONFIG["features"]:
    print(f"  {f}")
print(f"\nTarget: {CONFIG['target']}")

### Train/Test Split

We convert to pandas here — scikit-learn requires pandas DataFrames or numpy arrays, not Polars. The conversion happens at this **boundary** between our data pipeline (Polars) and our ML pipeline (sklearn).

In [ ]:
# Reload processed data and convert to pandas at the sklearn boundary
processed_df = pl.read_csv(DATA_DIR / "processed_data.csv").to_pandas()

X = processed_df[CONFIG["features"]]
y = processed_df[CONFIG["target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Features: {X_train.shape[1]}")
print("\nData leakage check: test set is held out from all tuning and training decisions.")

In [ ]:
# Save splits for recovery if kernel restarts
X_train.to_csv(DATA_DIR / "x_train.csv", index=False)
X_test.to_csv(DATA_DIR / "x_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)
print("Saved: x_train.csv, x_test.csv, y_train.csv, y_test.csv")

## Section 4: Model Training + Experiment Tracking

### Experiment Tracking with MLflow

Without tracking, ML experiments are invisible: "which model did I train? what params? what was the accuracy?" MLflow solves this by logging everything — params, metrics, data snapshots, and the model artifact — to a persistent store you can query and compare.

All three models run in a **single experiment** (`spotify-popularity`) so you can compare them side-by-side in the UI. Grids are intentionally smaller than production for classroom timing (~5 minutes total on Colab free tier).

In [ ]:
mlflow.set_experiment("spotify-popularity")

MODELS = {
    "LinearRegression": {
        "class": LinearRegression,
        "params": {"fit_intercept": [True, False]},
        "n_jobs": None,
    },
    "RandomForestRegressor": {
        "class": RandomForestRegressor,
        "params": {
            "n_estimators": [50, 100],
            "max_depth": [5, 10],
            "min_samples_split": [2, 5],
        },
        "n_jobs": -1,
    },
    "GradientBoostingRegressor": {
        "class": GradientBoostingRegressor,
        "params": {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
        },
        "n_jobs": None,  # GB is sequential — n_jobs has no effect
    },
}

print("Models to train:", list(MODELS.keys()))
print("\nGrid sizes (intentionally reduced from production for class timing):")
for name, cfg in MODELS.items():
    n = 1
    for v in cfg["params"].values():
        n *= len(v)
    print(f"  {name}: {n} combos x 5 folds = {n * 5} fits")

In [ ]:
results = []
best_run_id = None
best_r2 = -np.inf
best_model_name = None

for model_name, model_cfg in MODELS.items():
    print(f"\nTraining {model_name}...")

    estimator = model_cfg["class"]()
    grid_search = GridSearchCV(
        estimator,
        model_cfg["params"],
        cv=5,
        scoring="neg_mean_squared_error",
        n_jobs=model_cfg["n_jobs"],
        verbose=0,
    )

    with mlflow.start_run(run_name=model_name) as run:
        grid_search.fit(X_train, y_train)
        best_estimator = grid_search.best_estimator_

        y_pred = best_estimator.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

        # Log grid search results CSV as artifact
        cv_path = f"/tmp/{model_name}_cv_results.csv"
        pd.DataFrame(grid_search.cv_results_).to_csv(cv_path, index=False)
        mlflow.log_artifact(cv_path, "grid_search")

        # Log model artifact only (NOT registered here — registration happens in Section 5)
        input_example = X_train.head(5)
        signature = mlflow.models.infer_signature(X_train, y_pred)
        mlflow.sklearn.log_model(
            best_estimator,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
        )

        run_id = run.info.run_id

    results.append({"model": model_name, "rmse": rmse, "mae": mae, "r2": r2, "run_id": run_id})

    if r2 > best_r2:
        best_r2 = r2
        best_run_id = run_id
        best_model_name = model_name

    print(f"  RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}")

print(f"\nBest model: {best_model_name} (R2={best_r2:.3f})")
print(f"Best run ID: {best_run_id}")

In [ ]:
# Comparison table
results_df = pd.DataFrame(results).set_index("model").drop(columns=["run_id"])
print("\nModel Comparison (test set):")
print(results_df.round(4).to_string())
print("\nNext: open the MLflow UI, click 'spotify-popularity', select all 3 runs, and click Compare.")

### What to do in the MLflow UI

1. Open the URL printed in Section 0.4
2. Click the **spotify-popularity** experiment in the left sidebar
3. Select all 3 runs and click **Compare**
4. Check the **Metrics** tab — which model has the lowest RMSE and highest R2?
5. Check the **Parameters** tab to see the best hyperparameters per model
6. Click any run to see its artifacts (model files, grid search CSV)

## Section 5: Model Registry

### Model Registry: The End of "model_final_v2_FIXED.pkl"

A model registry is a central catalogue where models are versioned, annotated, and promoted to production. Without it, teams end up with files named `model_final_v2_FIXED_USE_THIS_ONE.pkl` with no record of what data it was trained on.

MLflow's Model Registry tracks:
- Every version of every model
- The run that produced it (linking back to training data, params, metrics)
- An **alias** indicating deployment status (`"production"` = the live model)

We register the single best model from Section 4 under a meaningful name: `"spotify-popularity-model"`.

In [ ]:
REGISTRY_MODEL_NAME = "spotify-popularity-model"

# Register best run's model artifact under our canonical model name
model_uri = f"runs:/{best_run_id}/model"
mv = mlflow.register_model(model_uri, REGISTRY_MODEL_NAME)

# Give the registry a moment to complete the registration
time.sleep(3)

print(f"Registered: {REGISTRY_MODEL_NAME} version {mv.version}")
print(f"From run: {best_run_id} ({best_model_name})")
print("\nIn the MLflow UI: click 'Models' in the left nav to see the registered model.")

### Setting an Alias

MLflow 2.9+ uses **aliases** instead of the deprecated Staging/Production stages. An alias is a named pointer to a specific version. Setting `"production"` as an alias means: "load whichever version has this alias." You can update which version production points to without changing any consuming code.

In [ ]:
client = MlflowClient()

# Set the "production" alias — this is the modern MLflow 2.x pattern
client.set_registered_model_alias(
    name=REGISTRY_MODEL_NAME,
    alias="production",
    version=mv.version,
)
print(f"Set alias 'production' -> version {mv.version}")

In [ ]:
# Load model back from registry using the alias
model = mlflow.sklearn.load_model(f"models:/{REGISTRY_MODEL_NAME}@production")
print(f"Loaded: {type(model).__name__}")

# Verify: predictions should match our Section 4 results
y_pred_registry = model.predict(X_test)

r2_verify = r2_score(y_test, y_pred_registry)
print(f"R2 from registry model: {r2_verify:.4f}")
print("\nSample prediction:")
print(f"  Input features: {X_test.iloc[0].to_dict()}")
print(f"  Predicted popularity: {y_pred_registry[0]:.1f}")
print(f"  Actual popularity: {y_test.iloc[0]}")

## Section 6: Inference + Data Drift Detection

### Inference: Using the Registered Model

Loading from the registry (not a file path) means inference code is decoupled from the model artifact. Deploying a new model version = updating the alias, not redeploying the service.

In [ ]:
# `model` and `REGISTRY_MODEL_NAME` are set in Section 5
# If you skipped Section 5, run: model = mlflow.sklearn.load_model("models:/spotify-popularity-model@production")
assert "model" in dir(), "Run Section 5 first to load the production model"

y_pred = model.predict(X_test)

# Log predictions as MLflow artifact
preds_df = pd.DataFrame({"actual": y_test.values, "predicted": y_pred.round(1)})
preds_path = "/tmp/predictions.csv"
preds_df.to_csv(preds_path, index=False)

with mlflow.start_run(run_name="inference"):
    mlflow.log_artifact(preds_path, "predictions")
    mlflow.log_metric("inference_r2", r2_score(y_test, y_pred))

print("Predictions logged.")
preds_df.head()

### Data Drift: Why Models Degrade

A model trained today may perform poorly next month — not because the model is wrong, but because the **input data distribution changed**. User listening habits shift, the data pipeline introduces noise, new genres emerge.

**Distribution shift**: the statistical distribution of features changes.
**Concept drift**: the relationship between features and the target changes.

We use [Evidently](https://docs.evidentlyai.com/) to detect distribution shift by running statistical tests per feature (Kolmogorov-Smirnov for numeric features).

In [ ]:
np.random.seed(42)
drifted_df = X_test.copy()

# Shift: imagine a trend toward higher-energy music
drifted_df["energy"] = (drifted_df["energy"] + 0.2).clip(upper=1.0)

# Noise: simulate a data pipeline rounding error worsening over time
drifted_df["tempo"] = drifted_df["tempo"] + np.random.normal(20, 5, len(drifted_df))

# Label noise: explicit flag getting flipped on ~10% of records
flip_mask = np.random.random(len(drifted_df)) < 0.1
drifted_df.loc[flip_mask, "explicit"] = 1 - drifted_df.loc[flip_mask, "explicit"]

print("Drift simulation:")
print(f"  energy mean:   {X_test['energy'].mean():.3f} -> {drifted_df['energy'].mean():.3f}")
print(f"  tempo mean:    {X_test['tempo'].mean():.1f} -> {drifted_df['tempo'].mean():.1f}")
print(f"  explicit mean: {X_test['explicit'].mean():.3f} -> {drifted_df['explicit'].mean():.3f}")

In [ ]:
# Evidently requires pandas DataFrames (X_train and drifted_df are both pandas)
report = Report([DataDriftPreset()])
result = report.run(reference_data=X_train, current_data=drifted_df)

# Save HTML and display inline
drift_html_path = "/tmp/drift_report.html"
result.save_html(drift_html_path)
display(HTML(open(drift_html_path).read()))

In [ ]:
# Log drift report as MLflow artifact
with mlflow.start_run(run_name="drift_detection"):
    mlflow.log_artifact(drift_html_path, "data_drift")

# Extract summary from report dict
drift_results = result.dump_dict()
try:
    mr = drift_results["metric_results"]
    for v in mr.values():
        if "Count of Drifted" in v["display_name"]:
            n_drifted = int(v["count"]["value"])
            share = v["share"]["value"]
            n_total = int(n_drifted / share) if share > 0 else 0
            break
    print("Drift summary:")
    print(f"  Features drifted: {n_drifted} / {n_total}")
    print(f"  Share of drifted columns: {share:.1%}")
except (KeyError, IndexError, ZeroDivisionError) as e:
    print(f"Could not parse drift summary: {e}")
    print("See the HTML report above for full results.")

### What To Do When Drift Is Detected

1. **Investigate the cause**: is it a data pipeline bug or genuine distribution shift?
2. **Retrain trigger**: if drift exceeds a threshold, kick off a new training run
3. **Model monitoring**: in production, run this check on every inference batch
4. **Alerting**: connect to PagerDuty/Slack when drift crosses thresholds

## Section 7: Model Serving (REST API)

### Model Serving: From Notebook to API

In production, models are rarely called directly from Python — they're served as REST APIs so any system (web apps, mobile apps, data pipelines) can request predictions over HTTP.

We create a minimal Flask endpoint that wraps `model.predict()`. We use an **in-process daemon thread** rather than `mlflow models serve` (which creates a separate virtualenv — slow and complex for Colab).

> The Flask server runs on the same Colab VM, so we reach it via `localhost`.

In [ ]:
# Guard: model must be loaded from Section 5/6
assert "model" in dir(), "Run Sections 5 and 6 first to load the production model"
assert "CONFIG" in dir(), "Run Section 3 first to define CONFIG"

serving_app = Flask(__name__)


@serving_app.route("/predict", methods=["POST"])
def predict():
    data = flask_request.get_json()
    df = pd.DataFrame(data["instances"])
    df = df[CONFIG["features"]]  # enforce column order
    predictions = model.predict(df)
    return jsonify({"predictions": predictions.tolist()})


@serving_app.route("/health")
def health():
    return "OK"


server_thread = threading.Thread(
    target=lambda: serving_app.run(port=7072, debug=False, use_reloader=False, threaded=True)
)
server_thread.daemon = True
server_thread.start()
print("Flask server starting on port 7072...")

In [ ]:
# Wait for server to be ready
for i in range(10):
    try:
        r = requests.get("http://localhost:7072/health", timeout=2)
        if r.status_code == 200:
            print("Server ready!")
            break
    except Exception:
        time.sleep(1)
else:
    print("Warning: server may not be ready yet")

### Single Prediction

In [ ]:
sample = X_test.iloc[[0]].to_dict(orient="records")

response = requests.post(
    "http://localhost:7072/predict",
    json={"instances": sample},
    headers={"Content-Type": "application/json"},
)

print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
print(f"\nDirect model prediction: {model.predict(X_test.iloc[[0]])[0]:.2f}")
print(f"API prediction:          {response.json()['predictions'][0]:.2f}")
print("(Should be identical — the API wraps the same model.predict() call)")

### Batch Prediction

In [ ]:
batch = X_test.iloc[:5].to_dict(orient="records")

response = requests.post(
    "http://localhost:7072/predict",
    json={"instances": batch},
    headers={"Content-Type": "application/json"},
)

predictions = response.json()["predictions"]
actuals = y_test.iloc[:5].values

comparison = pd.DataFrame(
    {
        "actual": actuals,
        "predicted": [round(p, 1) for p in predictions],
        "error": [round(abs(a - p), 1) for a, p in zip(actuals, predictions, strict=False)],
    }
)
print(comparison.to_string())

### How This Works in Production

In a real deployment, this Flask pattern would be replaced by:
- **MLflow Serving**: `mlflow models serve -m "models:/spotify-popularity-model@production" --port 7072`
- **FastAPI + Docker**: containerised service behind a load balancer
- **Cloud endpoints**: Azure ML, AWS SageMaker, Google Vertex AI

The key insight: the model is loaded from the **registry** (not a file path), and deploying a new version = updating the `"production"` alias — without changing application code.